# NBS x QMUL DEFEND Hackathon

You are the bank's fraud team. This notebook walks you through the job **step by step**, then hands over to you to improve it.

## The challenge

| File | Transactions | Do we know which are fraud? |
|---|---|---|
| `train.csv` | 10,000 | **Yes.** The `is_fraud` column tells us. |
| `test.csv` | 2,000 | **No.** This is what your team must work out. |

In the `is_fraud` column:
- **`1` = fraud**
- **`0` = genuine** (a normal transaction)

Your team must decide which transactions in `test.csv` are fraud. You will hand in a file where `is_flagged` is **`1` for the ones you think are fraud**, and `0` for the rest.

## Scoring

| What you do | Points |
|---|---|
| Flag a transaction that really is fraud | **+10** |
| Flag a genuine transaction | **-5** |
| Miss a fraud (you do not flag it) | **-10** |

#  Getting Started 

## Step 1: Load the Data

In [51]:
import numpy as np
import pandas as pd

train = pd.read_csv('train.csv')   # transactions with the answer column: is_fraud
test = pd.read_csv('test.csv')     # transactions without answers

print(f'train: {len(train):,} transactions     test: {len(test):,} transactions')

train: 10,000 transactions     test: 2,000 transactions


## Step 2: Explore at the data

Each row is one transaction. Run the cell to see the first five rows and how many transactions are fraud.

Explore some of the columns:
- `unknown_device`: 1 if the customer had not used this device before
- `failed_logins`: failed login attempts just before the transaction
- `new_payee`: 1 if money went to a recently added recipient
- `foreign_transaction`: 1 if it happened outside the UK, and `travel_notice` is 1 if the customer had told the bank about the trip
- `is_fraud`: **the answer**, 1 = fraud, 0 = genuine (only in `train.csv`)

The full list of columns is in `README.md`.

In [52]:
display(train.head())
print(train['is_fraud'].value_counts().rename({0: 'genuine', 1: 'fraud'}))

,transaction_id,timestamp,customer_id,account_type,typical_monthly_spend_gbp,transaction_type,amount_gbp,amount_vs_typical,hour_of_day,merchant_country,foreign_transaction,unknown_device,failed_logins,new_payee,travel_notice,is_fraud
0,TX-TRAIN-00001,2026-04-01 01:02:03,C122,STUDENT,4000,CARD_IN_STORE,25,0.01,1,GB,0,0,2,0,0,0
1,TX-TRAIN-00002,2026-04-01 01:59:58,C110,STUDENT,1000,CARD_ONLINE,150,0.15,1,GB,0,0,0,0,0,1
2,TX-TRAIN-00003,2026-04-01 06:31:41,C054,STUDENT,500,CARD_ONLINE,40,0.08,6,GB,0,1,0,0,0,0
3,TX-TRAIN-00004,2026-04-01 06:53:06,C032,STUDENT,500,CARD_ONLINE,40,0.08,6,ES,1,0,0,0,1,0
4,TX-TRAIN-00005,2026-04-01 07:07:37,C157,CURRENT,1000,CARD_ONLINE,40,0.04,7,GB,0,0,0,0,0,0


is_fraud
genuine    9200
fraud       800
Name: count, dtype: int64


## Step 3: Understand what fraud looks like

The table below shows the **average** of each column for genuine transactions (`0`) and fraud (`1`). 

**Think:** which three columns differ most? Could a genuine customer also have each of those signs? (A new phone, a holiday, a big one-off purchase...)

In [53]:
columns = ['amount_gbp', 'amount_vs_typical', 'hour_of_day', 'foreign_transaction', 'unknown_device',
           'failed_logins', 'new_payee', 'travel_notice']
train.groupby('is_fraud')[columns].mean().round(2).T

is_fraud,0,1
amount_gbp,105.19,1042.01
amount_vs_typical,0.11,1.11
hour_of_day,14.05,8.22
foreign_transaction,0.13,0.38
unknown_device,0.08,0.53
failed_logins,0.21,1.81
new_payee,0.02,0.36
travel_notice,0.08,0.00


### Looking at a specific column - unkown device

Reminder:
- `unknown_device`: 1 if the customer had not used this device before

In [54]:
# Change the column name and run again. Try 'new_payee', 'failed_logins', 'transaction_type' or 'hour_of_day'.
column = 'unknown_device'

(train.groupby(column)['is_fraud']
      .agg(transactions='count', share_that_is_fraud='mean')
      .round(2))

# Ask your own question: how often is a transaction fraud for each value of one column?

,transactions,share_that_is_fraud
unknown_device,,
0,8876,0.04
1,1124,0.38


## Step 4: How to use the dataset

**Practice set:** we know the answers for `train.csv`, so we keep 2,000 rows aside as a *practice set*. Nothing learns from them, so your score on them (Step 8) is an honest guide to how you might do on `test.csv`.

The cell below splits `train` into `learn` (8,000 rows to study) and `practice` (2,000 rows to test yourself).

In [55]:
from sklearn.model_selection import train_test_split

# learn    = 8,000 rows you can study
# practice = 2,000 rows kept aside to test yourself
learn, practice = train_test_split(train, test_size=2000, random_state=0)

print(f'learn: {len(learn):,} rows     practice: {len(practice):,} rows')

learn: 8,000 rows     practice: 2,000 rows


## Step 5: Your first rule

A **rule** is a yes/no test applied to every transaction. If the answer is yes, we flag it.

The rule below flags every transaction made from an unknown device. Run it to see how many it flags. Then change the rule and run it again. Ideas:
- `practice['failed_logins'] >= 3`
- `practice['new_payee'] == 1`
- `practice['amount_gbp'] >= 700`

You will see how well your rule scored in Step 8.

**Think:** why might a single rule flag genuine customers?

In [56]:
my_rule = practice['unknown_device'] == 1   # True means "flag this transaction"

print(f'Your rule flags {my_rule.sum()} of {len(practice):,} practice transactions')

Your rule flags 226 of 2,000 practice transactions


## Step 6: Combine rules into a score

One warning sign on its own is weak- what if a genuine customers is using a new phone? Several signs together are much stronger.

Below, each warning sign earns **1 point**, and a transaction is flagged when it reaches `THRESHOLD` points. The cell also lists the signs that fired as `reasons`, so an investigator can see why.

Try these, then check your score in Step 8:
1. Change `THRESHOLD` to 1, 2, then 3.
2. Remove a warning sign you think is weak.
3. Add your own sign (there is an example in the comments).

**Think:** which change do you expect to help most?

In [57]:
THRESHOLD = 2   # flag a transaction when it has at least this many warning signs

def warning_signs(tx):
    return pd.DataFrame({
        'Unknown device': tx['unknown_device'] == 1,
        '5+ failed logins': tx['failed_logins'] >= 5,
        'New payee': tx['new_payee'] == 1,
        'Amount 2000 or more': tx['amount_gbp'] >= 2000,
        # Add your own below, for example:
        # 'Big for this customer': tx['amount_vs_typical'] > 0.5,
    })

def score_rules(tx):
    signs = warning_signs(tx)
    out = tx[['transaction_id']].copy()
    out['risk_score'] = signs.sum(axis=1)
    out['reasons'] = signs.apply(lambda row: ' | '.join(signs.columns[row.to_numpy()]), axis=1)
    out['is_flagged'] = (out['risk_score'] >= THRESHOLD).astype(int)
    return out

flagged = score_rules(practice)['is_flagged'].sum()
print(f'Your rules flag {flagged} of {len(practice):,} practice transactions')

Your rules flag 51 of 2,000 practice transactions


## Step 7: Try a simple machine learning model

In Step 6 *you* chose the rules. A machine learning model can find rules from the data instead. A **decision tree** is about the simplest model there is: it learns a few yes/no questions from the `learn` rows.

Run the cell. It will:
1. train the tree,
2. print the questions it found (read them like rules!).

Each answer gives a fraud **probability**, and a transaction is flagged when its probability is at least `ML_THRESHOLD`. A low threshold flags more transactions, and a high one flags fewer.

The model will not be perfect, and that is fine. You will compare it with your rules in Step 8. **Think:** can you explain how this works to the judges?

In [58]:
from sklearn.tree import DecisionTreeClassifier, export_text

ML_THRESHOLD = 0.3   # flag when the model's fraud probability is at least this

FEATURES = ['amount_gbp', 'amount_vs_typical', 'hour_of_day', 'foreign_transaction', 'unknown_device',
            'failed_logins', 'new_payee', 'travel_notice']

def make_X(tx):
    X = tx[FEATURES].copy()
    X['is_transfer'] = (tx['transaction_type'] == 'BANK_TRANSFER').astype(int)
    X['is_online'] = (tx['transaction_type'] == 'CARD_ONLINE').astype(int)
    return X

model = DecisionTreeClassifier(max_depth=3, random_state=0)   # max_depth = how many questions deep
model.fit(make_X(learn), learn['is_fraud'])
print(export_text(model, feature_names=list(make_X(learn).columns)))

def score_ml(tx):
    prob = model.predict_proba(make_X(tx))[:, 1]
    out = tx[['transaction_id']].copy()
    out['risk_score'] = (prob * 100).round(1)
    out['reasons'] = score_rules(tx)['reasons']
    out['is_flagged'] = (prob >= ML_THRESHOLD).astype(int)
    return out

flagged = score_ml(practice)['is_flagged'].sum()
print(f'Your model flags {flagged} of {len(practice):,} practice transactions')

|--- amount_gbp <= 475.00
|   |--- failed_logins <= 3.50
|   |   |--- hour_of_day <= 5.50
|   |   |   |--- class: 0
|   |   |--- hour_of_day >  5.50
|   |   |   |--- class: 0
|   |--- failed_logins >  3.50
|   |   |--- class: 1
|--- amount_gbp >  475.00
|   |--- failed_logins <= 0.50
|   |   |--- amount_gbp <= 1250.00
|   |   |   |--- class: 0
|   |   |--- amount_gbp >  1250.00
|   |   |   |--- class: 1
|   |--- failed_logins >  0.50
|   |   |--- class: 1

Your model flags 99 of 2,000 practice transactions


## Step 8: Score yourself

Now see how well your rule, your combined rules and your model did on the practice set.

| Event | Points |
|---|---|
| You flag a transaction that is fraud | **+10** |
| You flag a genuine transaction | **-5** |
| You miss a fraud (you do not flag it) | **-10** |

Flagging everything is a disaster, because most transactions are genuine. Flagging nothing is a loss too, because every missed fraud costs the bank. So flag a transaction when there is about a **1 in 5** chance, or better, that it is fraud.

Run the cell below. **Come back and re-run it after every change you make in Steps 5 to 7.** The last part shows how the model scores at different thresholds.

In [59]:
def show_points(flags, data):
    # flags = True for each transaction you flag, False for the rest
    flags = pd.Series(np.asarray(flags), index=data.index).astype(bool)
    is_fraud = data['is_fraud'] == 1

    caught = (flags & is_fraud).sum()          # flagged, and really fraud
    false_alarms = (flags & ~is_fraud).sum()   # flagged, but genuine
    missed = (~flags & is_fraud).sum()         # fraud you did not flag

    points = 10 * caught - 5 * false_alarms - 10 * missed
    print(f'Points: {points}   (caught {caught} fraud, {false_alarms} false alarms, missed {missed})')

print('Step 5, your first rule:')
show_points(my_rule, practice)

print('\nStep 6, combined rules:')
show_points(score_rules(practice)['is_flagged'], practice)

print('\nStep 7, model:')
show_points(score_ml(practice)['is_flagged'], practice)

print('\nStep 7, model at different thresholds:')
practice_prob = model.predict_proba(make_X(practice))[:, 1]
for t in [0.2, 0.3, 0.4, 0.5, 0.6]:
    print(f'  threshold {t}:', end='  ')
    show_points(practice_prob >= t, practice)

Step 5, your first rule:
Points: -650   (caught 80 fraud, 146 false alarms, missed 72)

Step 6, combined rules:
Points: -600   (caught 47 fraud, 4 false alarms, missed 105)

Step 7, model:
Points: 460   (caught 99 fraud, 0 false alarms, missed 53)

Step 7, model at different thresholds:
  threshold 0.2:  Points: 615   (caught 125 fraud, 73 false alarms, missed 27)
  threshold 0.3:  Points: 460   (caught 99 fraud, 0 false alarms, missed 53)
  threshold 0.4:  Points: 460   (caught 99 fraud, 0 false alarms, missed 53)
  threshold 0.5:  Points: 460   (caught 99 fraud, 0 false alarms, missed 53)
  threshold 0.6:  Points: 460   (caught 99 fraud, 0 false alarms, missed 53)


## Your turn

You now have a working rules score (Step 6) and a simple model (Step 7). Make one of them better, or build something new. Change **one thing at a time**, re-run Step 8 to see your score, and keep a log:

| What I changed | Points before | Points after | Keep it? Why? |
|---|---|---|---|
| | | | |

**Ideas for rules**
- Add a sign using a column you have not used yet (`amount_vs_typical`, `account_type`, `transaction_type`).
- Stop flagging foreign payments when `travel_notice` is 1.
- Give stronger signs 2 points instead of 1 (change `warning_signs` to add weights).

**Ideas for the model**
- Change `max_depth` in Step 7 (try 2, 5 or 8). Does deeper always help?
- Add features to `FEATURES`, or build a new one in `make_X`.
- Swap in another model, for example `RandomForestClassifier` or `LogisticRegression`, from `sklearn`.

**Ideas for exploring**
- Draw a chart that reveals a pattern, for example `train.groupby('hour_of_day')['is_fraud'].mean().plot.bar()`.
- Combine the model's score with your rules.

**Remember to discuss:** how many alerts could the bank realistically review, and why did you choose your threshold?

## Final step: Predict on `test.csv` and hand in your answers

`test.csv` has 2,000 new transactions with **no answers**. Your job now is to say, for each one, whether you think it is fraud.

The cell below:
1. takes the approach you choose (`FINAL`) and applies it to every transaction in `test.csv`,
2. creates a **new file**, `test_predictions.csv`, with one row per transaction,
3. gives each transaction three columns: `transaction_id`, `reasons` and `is_flagged`.

`is_flagged` is **1 if you think it is fraud, 0 if you think it is genuine**, and `reasons` says why you flagged it.

The file looks like this:

| transaction_id | reasons | is_flagged |
|---|---|---|
| TX-TEST-00001 | | 0 |
| TX-TEST-00002 | Unknown device \| 3+ failed logins | 1 |

You will not see a score, because the answers to `test.csv` are kept by the organisers. They compare your `is_flagged` column with the real answers.

**To finish:** set `FINAL` to the approach your team chose, run the cell, check the preview, then hand in `test_predictions.csv`. Your original `test.csv` is not changed.

In [60]:
FINAL = score_ml   # your team's approach: score_ml, score_rules, or your own function

# Keep only the three columns we need: which transaction, why, and the fraud call (1 or 0).
predictions = FINAL(test)[['transaction_id', 'reasons', 'is_flagged']]
predictions.to_csv('test_predictions.csv', index=False)

print(f'Saved test_predictions.csv: {len(predictions):,} transactions, {predictions.is_flagged.sum()} flagged as fraud')
predictions.head()

Saved test_predictions.csv: 2,000 transactions, 112 flagged as fraud


,transaction_id,reasons,is_flagged
0,TX-TEST-00001,New payee,1
1,TX-TEST-00002,,0
2,TX-TEST-00003,,0
3,TX-TEST-00004,,0
4,TX-TEST-00005,,0
